# CP3-08 检查点分支 Fork：修改过去，长出新的未来

本节使用 `InMemorySaver` 和结构化路由图，重点观察 `update_state` 的 `as_node` 参数。先从同一个 `thread_id` 取得路由节点之前的检查点，再从这个锚点生成两条不同的未来。

## Replay 与 Fork

- Replay：不修改状态，从旧 checkpoint 原样续跑。
- Fork：用 `update_state` 写入新状态，生成新的 `checkpoint_id`；新快照的 `parent_config` 指向旧锚点，原来的历史仍保留。

`with_structured_output` 让模型返回符合 schema 的字典；`mode` 使用 `Literal` 枚举，`router` 再把它转成确定的业务标签。`Annotated` 中的中文说明是字段描述，不是 Reducer。若当前模型网关不支持结构化输出，可把该节点降级为普通文本输出，再用 Python 手动解析并映射 `mode`。


In [ ]:
import os
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from loguru import logger

# 本节使用内存检查点，不依赖 PostgreSQL；但 Python 进程结束后历史会消失。
load_dotenv(override=True)
MODEL_NAME = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
if not os.getenv('DEEPSEEK_API_KEY'):
    raise RuntimeError('缺少 DEEPSEEK_API_KEY，请先配置 .env 或系统环境变量。')
model = ChatDeepSeek(model=MODEL_NAME, extra_body={'thinking': {'type': 'disabled'}})

# topic/mode 是路由节点写入的内部字段，显式放入全局 State，避免依赖宽松合并。
class OverAllState(TypedDict, total=False):
    username: str
    user_input: str
    topic: str
    mode: str
    output: str

# 结构化输出 schema 约束模型返回主题与合法模式。
class StructuredOutputState(TypedDict):
    topic: Annotated[str, '主题']
    mode: Annotated[Literal['poem', 'joke'], '模式']

model_with_structure = model.with_structured_output(schema=StructuredOutputState)

# router_node 负责让模型理解意图；它不直接决定图的物理下一节点。
def router_node(state: OverAllState) -> StructuredOutputState:
    logger.info('路由节点已经执行')
    result = model_with_structure.invoke([HumanMessage(content=state['user_input'])])
    logger.info('路由结果为: {}', result)
    return result

# router 是纯 Python 路由函数，把结构化 mode 映射为业务标签。
def router(state: OverAllState) -> Literal['poem', 'joke', 'default']:
    logger.info('路由函数已经执行')
    if state.get('mode') == 'poem':
        logger.info('路由至 node_poem 节点')
        return 'poem'
    if state.get('mode') == 'joke':
        logger.info('路由至 node_joke 节点')
        return 'joke'
    logger.info('路由到兜底节点')
    return 'default'

# 下游节点将模型响应的 content 存为字符串，便于序列化和展示。
def node_poem(state: OverAllState) -> OverAllState:
    logger.info('node_poem 已执行')
    response = model.invoke([HumanMessage(content=f'写一首关于{state["topic"]}主题的七言绝句，不要赏析，只写诗句')])
    return {'output': response.content}

def node_joke(state: OverAllState) -> OverAllState:
    logger.info('node_joke 已执行')
    response = model.invoke([HumanMessage(content=f'写一个关于{state["topic"]}主题的笑话，字数在100以内')])
    return {'output': response.content}

def node_default(state: OverAllState) -> OverAllState:
    logger.info('node_default 已执行')
    return {'output': '无法处理的任务类型'}

# path_map 把业务标签和物理节点名解耦，后续重命名节点只需改映射。
builder = StateGraph(state_schema=OverAllState)
builder.add_node('router_node', router_node)
builder.add_node('node_poem', node_poem)
builder.add_node('node_joke', node_joke)
builder.add_node('node_default', node_default)
builder.add_edge(START, 'router_node')
builder.add_conditional_edges(
    'router_node',
    router,
    path_map={'poem': 'node_poem', 'joke': 'node_joke', 'default': 'node_default'},
)
builder.add_edge('node_poem', END)
builder.add_edge('node_joke', END)
builder.add_edge('node_default', END)

checkpointer = InMemorySaver()
THREAD_ID = os.getenv('CP3_FORK_THREAD_ID', 'chapter03-08-fork')
config = {'configurable': {'thread_id': THREAD_ID}}
graph = builder.compile(checkpointer=checkpointer)
print(graph.get_graph().draw_mermaid())

first_result = graph.invoke(
    {'username': '小王', 'user_input': '写一首关于荷花的诗'},
    config=config,
)
print(first_result)


## 先找到分支锚点

历史通常按新到旧排列。`next=('router_node',)` 表示起点输入已写入，但路由节点还没有执行；从这个快照分支，可以比较“改输入”和“改路由产出”两种策略。


In [ ]:
history_checkpoints = list(graph.get_state_history(config=config))
if not history_checkpoints:
    raise RuntimeError('没有找到当前 thread_id 的历史检查点。')

before_router_checkpoint = next(
    (snapshot for snapshot in history_checkpoints if snapshot.next == ('router_node',)),
    None,
)
if before_router_checkpoint is None:
    raise RuntimeError('没有找到 router_node 之前的检查点。')

print({
    'checkpoint_id': before_router_checkpoint.config['configurable'].get('checkpoint_id'),
    'next': before_router_checkpoint.next,
    'values': before_router_checkpoint.values,
})


## 分支方式 A：修改输入，重新走路由

把更新标记为 `as_node=START`，意味着这些值被视为起点产生的新输入。因此新快照的 `next` 仍然是 `router_node`，续跑时会重新调用结构化路由模型，再进入笑话节点。


In [ ]:
change_input_config = graph.update_state(
    config=before_router_checkpoint.config,
    values={'user_input': '帮我写一个荷花的笑话'},
    as_node=START,
)
print(change_input_config)
print(graph.get_state(change_input_config))

# input=None 表示从这个分支快照续跑，router_node 会重新调用模型。
result_after_input_fork = graph.invoke(None, config=change_input_config)
print(result_after_input_fork)


## 分支方式 B：伪造节点输出，跳过一次模型调用

这次告诉 LangGraph：`topic` 和 `mode` 就是 `router_node` 的产出。于是 `router_node` 本身不会再调用 LLM，但它的条件出边仍会根据新 `mode` 重算，新的 `next` 会直接指向 `node_joke`。跳过模型节点不等于跳过路由函数。


In [ ]:
skip_router_config = graph.update_state(
    config=before_router_checkpoint.config,
    values={'topic': '狸花猫', 'mode': 'joke'},
    as_node='router_node',
)
print(skip_router_config)
print(graph.get_state(skip_router_config))

# 这里只执行 node_joke；output 已统一保存为字符串。
result_after_output_fork = graph.invoke(None, config=skip_router_config)
print(result_after_output_fork)


## 两种 Fork 的对照结论

| 方式 | as_node | 修改字段 | 是否重跑 router_node | 新快照 next |
| --- | --- | --- | --- | --- |
| 改输入 | START | user_input | 是 | router_node |
| 改节点产出 | router_node | topic、mode | 否 | node_joke 或 node_poem |

每次 `update_state` 都会创建新的检查点，而不是改写旧快照。这个性质让人工审核、纠错、回滚和“如果当时选择另一条路径”类产品功能成为可能。
